# 🔀 HCVRPTW Multi-Decoder Step-by-Step Training
Notebook ini membongkar logika dari **`run_multi.py`**. 
Multi-Decoder adalah versi paling *advanced* (SOTA) di mana model dipaksa menghasilkan beberapa kandidat rute sekaligus (`n_paths`) dan dilatih menggunakan pinalti **Kullback-Leibler (KL) Divergence** agar rute-rute kandidat tersebut bervariasi (*diverse*).

In [ ]:
# 1. Setup Environment
# !git clone https://github.com/your-repo/routing-optim.git
# %cd routing-optim/fleet_v3_gcollab
# !pip install -r ../requirements.txt

In [1]:
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import math

# Fallback jika jupyter/ipywidgets tidak terinstal
try:
    import ipywidgets
    from tqdm.notebook import tqdm
    tqdm(range(0)).close()
except (ImportError, NameError, Exception):
    from tqdm import tqdm

# Import dari struktur internal fleet_v3
from core.problems import HCVRP
from core.nets.attention_model import AttentionModelMultiDecoder
from core.reinforce_baselines import RolloutBaseline
from utils import move_to
from core.nets.attention_model import set_decode_type

## ⚙️ 1. Konfigurasi Multi-Decoder
Perhatikan bahwa `opts.n_paths = 5` (jumlah decoder) dan `opts.kl_loss = 0.1` (koefisien pinalti untuk mencegah rute yang terlalu mirip).

In [2]:
import time
class DummyOptions:
    def __init__(self):
        # Default values dari options.py untuk mencegah AttributeError
        self._defaults = {
            'problem': 'hcvrp',
            'graph_size': 20,
            'batch_size': 512,
            'epoch_size': 1280000,
            'val_size': 10000,
            'val_dataset': None,
            'obj': 'min-max',
            'model': 'attention_multi_decoder',
            'embedding_dim': 128,
            'hidden_dim': 128,
            'n_encode_layers': 3,
            'tanh_clipping': 10.,
            'normalization': 'batch',
            'lr_model': 1e-4,
            'lr_critic': 1e-4,
            'lr_decay': 0.995,
            'eval_only': False,
            'n_epochs': 50,
            'seed': 1234,
            'max_grad_norm': 3.0,
            'no_cuda': False,
            'exp_beta': 0.8,
            'baseline': None,
            'bl_alpha': 0.05,
            'bl_warmup_epochs': None,
            'eval_batch_size': 1024,
            'checkpoint_encoder': False,
            'shrink_size': None,
            'data_distribution': None,
            'log_step': 50,
            'log_dir': 'logs',
            'run_name': 'run',
            'output_dir': 'outputs',
            'save_dir': 'outputs/hcvrp_20/run',
            'epoch_start': 0,
            'checkpoint_epochs': 1,
            'load_path': None,
            'resume': None,
            'no_tensorboard': False,
            'no_progress_bar': False,
            'n_paths': 5,
            'kl_loss': 1.0,
        }

    def __getattr__(self, name):
        if name in self._defaults:
            return self._defaults[name]
        raise AttributeError(f"'DummyOptions' object has no attribute '{name}'")

opts = DummyOptions()

# =====================================================================
# SELECT PRESET CONFIGURATION (Cukup ganti nilai SELECTED_PRESET di bawah)
# Pilihan:
#   - 'run_testing_multi'   (graph_size=10, 5 paths, debug)
#   - 'train_multi_decoder' (graph_size=20, 5 paths, matching Makefile)
# =====================================================================
SELECTED_PRESET = 'train_multi_decoder' 

PRESETS = {
    'run_testing_multi': {
        'graph_size': 10,
        'batch_size': 3,
        'epoch_size': 3,
        'val_size': 2,
        'n_epochs': 3,
        'lr_model': 1e-4,
        'lr_critic': 1e-4,
        'lr_decay': 0.99,
        'max_grad_norm': 1.0,
        'embedding_dim': 32,
        'hidden_dim': 32,
        'n_encode_layers': 2,
        'tanh_clipping': 10.0,
        'normalization': 'batch',
        'obj': 'min-max',
        'model': 'attention_multi_decoder',
        'baseline': 'rollout',
        'n_paths': 5,
        'kl_loss': 1.0,
        'checkpoint_encoder': False,
        'shrink_size': None,
        'data_distribution': None,
    },
    'train_multi_decoder': {
        'graph_size': 20,
        'batch_size': 512,
        'epoch_size': 1280000,
        'val_size': 10000,
        'n_epochs': 50,
        'lr_model': 1e-4,
        'lr_critic': 1e-4,
        'lr_decay': 0.995,
        'max_grad_norm': 3.0,
        'embedding_dim': 128,
        'hidden_dim': 128,
        'n_encode_layers': 3,
        'tanh_clipping': 10.0,
        'normalization': 'batch',
        'obj': 'min-max',
        'model': 'attention_multi_decoder',
        'baseline': 'rollout',
        'n_paths': 5,
        'kl_loss': 1.0,
        'checkpoint_encoder': False,
        'shrink_size': None,
        'data_distribution': None,
    }
}

# Terapkan konfigurasi terpilih
config = PRESETS[SELECTED_PRESET]
for key, val in config.items():
    setattr(opts, key, val)

# Setup output directory & run_name secara otomatis
opts.run_name = f"{opts.model}_colab_{time.strftime('%Y%m%dT%H%M%S')}"
opts.save_dir = os.path.join(opts.output_dir, f"{opts.problem}_{opts.graph_size}", opts.run_name)
os.makedirs(opts.save_dir, exist_ok=True)

# Hardware & Evaluasi
opts.use_cuda = torch.cuda.is_available()
opts.device = torch.device('cuda:0' if opts.use_cuda else 'cpu')
opts.eval_batch_size = 4

print(f"Menggunakan Preset: {SELECTED_PRESET}")
print(f"Output directory untuk simpan checkpoint: {opts.save_dir}")
print(f"Device: {opts.device} | Model: {opts.model} | Graph Size: {opts.graph_size}")


Menggunakan Preset: train_multi_decoder
Output directory untuk simpan checkpoint: outputs\hcvrp_20\attention_multi_decoder_colab_20260603T133115
Device: cpu | Model: attention_multi_decoder | Graph Size: 20


## 🤖 2. Inisialisasi Environment & Model

In [3]:
problem = HCVRP()

# Inisialisasi Model Multi-Decoder
model = AttentionModelMultiDecoder(
    opts.embedding_dim,
    opts.hidden_dim,
    opts.obj,
    problem,
    n_encode_layers=opts.n_encode_layers,
    mask_inner=True,
    mask_logits=True,
    normalization=opts.normalization,
    tanh_clipping=opts.tanh_clipping,
    checkpoint_encoder=opts.checkpoint_encoder,
    shrink_size=opts.shrink_size,
    n_paths=opts.n_paths,
).to(opts.device)

baseline = RolloutBaseline(model, problem, opts)
optimizer = optim.Adam([{'params': model.parameters(), 'lr': opts.lr_model}])
lr_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda epoch: opts.lr_decay ** epoch)

print("Model Multi-Decoder Siap.")

Evaluating baseline model on evaluation dataset


  5%|▍         | 119/2500 [01:14<24:45,  1.60it/s]


KeyboardInterrupt: 

## 🔬 3. Training Loop Khusus Multi-Decoder
Perhatikan bagian pengecekan `pack[3]`. Pada *multi-decoder*, model tidak hanya mengembalikan 1 nilai *cost*, tapi dia mengembalikan *matrix* probabilitas untuk 5 jalur berbeda beserta perhitungan pinalti KL (*Kullback-Leibler*).

In [ ]:
epoch_costs = []
epoch_losses = []

for epoch in range(opts.n_epochs):
    print(f"\n=== Mulai Epoch {epoch} ===")
    lr_scheduler.step(epoch)
    
    training_dataset = baseline.wrap_dataset(problem.make_dataset(
        size=opts.graph_size, num_samples=opts.epoch_size, distribution=opts.data_distribution
    ))
    training_dataloader = DataLoader(training_dataset, batch_size=opts.batch_size, num_workers=0)
    
    model.train()
    set_decode_type(model, "sampling")
    
    epoch_cost_sum = 0
    epoch_loss_sum = 0
    num_batches = len(training_dataloader)
    
    for batch_id, batch in enumerate(tqdm(training_dataloader, desc=f"Epoch {epoch}")):
        x, bl_val = baseline.unwrap_batch(batch)
        x = move_to(x, opts.device)
        bl_val = move_to(bl_val, opts.device) if bl_val is not None else None
        
        # --- FORWARD PASS MULTI-DECODER ---
        # Untuk Multi-Decoder, 'out' menghasilkan tuple of 4!
        out = model(x, opts=opts)
        
        cost = out[0] # Cost minimum dari n_paths
        log_likelihood = out[1]
        log_veh = out[2]
        pack = out[3] # Matrix berukuran 4 yang berisi cost, LL, LL_veh, dan pinalti KL
        
        # Membongkar 'pack' untuk menghitung pinalti KL
        costs_p, ll_p, llv_p = pack[0], pack[1], pack[2]
        
        # Baseline Eval
        bl_val, bl_loss = baseline.eval(x, cost) if bl_val is None else (bl_val, 0)
        
        # Perhitungan Loss (REINFORCE + KL Divergence)
        reinforce_loss = ((costs_p - bl_val) * (ll_p + llv_p)).mean()
        
        # Ekstrak KL Penalty term
        kl_term = -opts.kl_loss * pack[3].mean()
        loss = reinforce_loss + bl_loss + kl_term
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), opts.max_grad_norm)
        optimizer.step()
        
        epoch_cost_sum += cost.mean().item()
        epoch_loss_sum += loss.item()
        
        if batch_id == 0:
            print(f"[Batch 0] Min Cost: {cost.mean().item():.3f} | Total Loss: {loss.item():.3f} | KL Penalty: {kl_term.item():.3f}")
            
    baseline.epoch_callback(model, epoch)
    
    # Simpan rata-rata cost dan loss per epoch
    epoch_costs.append(epoch_cost_sum / num_batches)
    epoch_losses.append(epoch_loss_sum / num_batches)
    
    # --- SIMPAN CHECKPOINT MODEL ---
    checkpoint_path = os.path.join(opts.save_dir, f'epoch-{epoch}.pt')
    print(f"Menyimpan checkpoint model ke: {checkpoint_path}")
    torch.save(
        {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'rng_state': torch.get_rng_state(),
            'baseline': baseline.state_dict()
        },
        checkpoint_path
    )


In [ ]:
# --- Visualisasi Hasil Training ---
import matplotlib.pyplot as plt

epochs = range(len(epoch_costs))

plt.figure(figsize=(14, 5))

# Plot Training Cost (Average Min Cost)
plt.subplot(1, 2, 1)
plt.plot(epochs, epoch_costs, color='dodgerblue', marker='o', linewidth=2, label='Average Min Cost')
plt.title('Training Cost (Average Min Distance) per Epoch', fontsize=12, weight='bold')
plt.xlabel('Epoch', fontsize=10)
plt.ylabel('Cost (Distance)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Plot Training Loss (REINFORCE + KL Loss)
plt.subplot(1, 2, 2)
plt.plot(epochs, epoch_losses, color='crimson', marker='s', linewidth=2, label='Average Loss')
plt.title('Training Loss per Epoch', fontsize=12, weight='bold')
plt.xlabel('Epoch', fontsize=10)
plt.ylabel('Loss (REINFORCE + KL)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()
